# Production RAG Pipeline — Customer Support Domain (techqa / emanual / delucionqa)

This notebook runs the **final, chosen production configuration** end-to-end across all three
customer-support subsets of `galileo-ai/ragbench`, and reports its TRACe scores. It is *not* an
ablation sweep — that work is in `rag-experiments/{dataset}-openrouter-experiment/` — this notebook
exists to demonstrate and validate the single configuration that came out of that sweep as the winner.

## Why this configuration

After a full ablation sweep (up to 11 variants per dataset: baseline, embedder swap, chunking swap,
hybrid dense+sparse fusion with and without reranking, HyDE, step-back, combined-lever configs, and an
externally-inspired wide-retrieval variant) and a rigorous n=100 head-to-head validation on the two
strongest contenders, the plain **dense-only retrieval baseline** won:

- **Chunking**: fixed-word, sized per dataset (128/200/160 words for emanual/techqa/delucionqa)
- **Embedding**: `sentence-transformers/all-MiniLM-L6-v2` (384d) — beat the stronger `BAAI/bge-base-en-v1.5`
  once sample size went beyond n=20
- **Retrieval**: dense-only, top_k=5 — no hybrid fusion, no reranking, no query transform
- **Generation/Judge**: a large capable model (`meta-llama/llama-3.3-70b-instruct` here; `llama-3.3-70b-versatile`
  on Groq) — confirmed to matter far more than any retrieval-side lever, both in our own techqa run and
  independently in a different team's capstone pipeline using a smaller 8B generator

Every attempt to beat this with something fancier either underperformed in the original sweep or lost
the larger-sample validation. See `production/configs/*.yaml` for the per-dataset rationale, and
`production/run_pipeline.py` for a standalone reusable runner (not tied to RAGBench/notebooks).

**Known open limitation, not fixed by this or any config tried**: techqa's real (non-refusal) adherence
stays near-zero regardless of retrieval strategy — a generation-prompt/domain issue, not a retrieval
problem, and worth separate follow-up work.

## 1. Setup & Dependencies

In [ ]:
get_ipython().system('pip3 install datasets faiss-cpu sentence-transformers torch groq openai python-dotenv nltk pandas rank_bm25 -q')

## 2. Imports & Project Root

In [ ]:
import sys
import os
from pathlib import Path
from dotenv import load_dotenv

# --- Point this at wherever THIS repo (rag_cust_support) lives. ---
# On Colab this is typically under your mounted Drive. Adjust if different.
PROJECT_ROOT = Path('/content/drive/MyDrive/Capstone/rag_cust_support')
if not PROJECT_ROOT.exists():
    # Fallback: running locally from the notebooks/ folder.
    PROJECT_ROOT = Path.cwd().parent if 'notebooks' in str(Path.cwd()) else Path.cwd()

os.chdir(PROJECT_ROOT)
project_root = PROJECT_ROOT
# Make THIS repo win on sys.path (avoids importing a stale rag-foundry copy).
sys.path = [p for p in sys.path if 'rag-foundry' not in p]
if str(project_root) in sys.path:
    sys.path.remove(str(project_root))
sys.path.insert(0, str(project_root))

from experiment.experiment_config import ExperimentConfig
from experiment.experiment_runner import ExperimentRunner
import experiment.experiment_runner as _er, core.registry as _reg

load_dotenv(override=True)
print('Current directory:', Path.cwd())
print('experiment_runner loaded from:', _er.__file__)
assert 'rag-foundry' not in _er.__file__, 'Still importing the old rag-foundry code! Restart runtime.'
print('HuggingFace token loaded:', bool(os.getenv('HF_TOKEN')))
print('Groq API key loaded:', bool(os.getenv('GROQ_API_KEY')))
print('OpenRouter API key loaded:', bool(os.getenv('OPENROUTER_API_KEY')))

## 3. Run the production config for each dataset

Each dataset has its own isolated experiment dir (`rag-experiments/{dataset}-production/`) containing
**only** the one chosen production config — not the full ablation sweep — driven by
`experiment_configs/{dataset}_production_experiment.yaml`. Defaults to `end_index=20` per dataset for a
quick validation pass; raise it (up to each dataset's full test-split size) for a fuller production
validation once you're ready — the run is checkpoint-resumable, so raising it later won't re-cost
anything already completed.

In [ ]:
DATASETS = ['techqa', 'emanual', 'delucionqa']

runners = {}
reports = {}

for dataset in DATASETS:
    print(f'\n{"="*80}\n{dataset.upper()} — production config\n{"="*80}')

    experiment_config = ExperimentConfig.load(project_root / f'experiment_configs/{dataset}_production_experiment.yaml')
    runner = ExperimentRunner(experiment_config)
    runners[dataset] = runner

    documents, raw_data = runner.load_data()
    print(f'Loaded {len(raw_data)} raw rows -> {len(documents)} parsed documents')

    configs = runner.load_configs()
    assert len(configs) == 1, f'Expected exactly 1 production config for {dataset}, found {len(configs)}'
    print(f'Running production config: {configs[0].name}')

    runs = runner.run(documents, raw_data)
    runs = runner.evaluate_runs(runs)
    reports[dataset] = runner.generate_reports(runs)

## 4. Results

In [ ]:
import pandas as pd
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 80)

for dataset in DATASETS:
    print(f'\n{"="*80}\n{dataset.upper()}\n{"="*80}')
    comparison = runners[dataset].compare()
    display(comparison.to_dataframe())

## 5. Conclusion

This is the configuration recommended for the customer-support domain going forward. It is deliberately
the *simplest* option in the whole sweep — no hybrid fusion, no reranker, no query transform — which won
specifically because every added layer of retrieval sophistication either underperformed in the original
sweep or lost the head-to-head validation against it. For real reuse outside this benchmark/notebook
context (e.g. against your own documents), see `production/run_pipeline.py`.